In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ALPHA_API_KEY")

BASE = "https://www.alphavantage.co/query"
SYMBOLS = ["AMD", "ARM", "NVDA", "INTC", "MU"]
OUT_DIR = "data"
RATE_LIMIT_SLEEP = 1  # saniye

def call_av(params, max_retries=3):
    params["apikey"] = API_KEY
    for attempt in range(max_retries):
        r = requests.get(BASE, params=params, timeout=40)
        if r.status_code == 200:
            data = r.json()
            if "quarterlyReports" in data:
                return data
            if "Note" in data or "Error Message" in data:
                time.sleep(2 + attempt)
                continue
        else:
            time.sleep(2)
    return {}

def f(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def fetch_full_history(symbol):
    print(f"📥 {symbol} tüm çeyrekler alınıyor...")
    inc = call_av({"function": "INCOME_STATEMENT", "symbol": symbol})
    bal = call_av({"function": "BALANCE_SHEET", "symbol": symbol})
    cf  = call_av({"function": "CASH_FLOW", "symbol": symbol})
    ov  = call_av({"function": "OVERVIEW", "symbol": symbol})

    inc_q = inc.get("quarterlyReports", [])
    bal_q = bal.get("quarterlyReports", [])
    cf_q  = cf.get("quarterlyReports", [])

    fiscal_dates = sorted({
        rep["fiscalDateEnding"] for lst in [inc_q, bal_q, cf_q] for rep in lst if rep.get("fiscalDateEnding")
    })

    records = []
    for date in fiscal_dates:
        incp = next((x for x in inc_q if x.get("fiscalDateEnding") == date), {})
        balp = next((x for x in bal_q if x.get("fiscalDateEnding") == date), {})
        cfp  = next((x for x in cf_q if x.get("fiscalDateEnding") == date), {})

        rev = f(incp.get("totalRevenue"))
        net = f(incp.get("netIncome"))
        gross = f(incp.get("grossProfit"))
        op_inc = f(incp.get("operatingIncome"))
        ebitda = f(incp.get("ebitda") or (ov.get("EBITDA") if isinstance(ov, dict) else np.nan))
        ocf = f(cfp.get("operatingCashflow"))
        capex = f(cfp.get("capitalExpenditures"))
        fcf = f(cfp.get("freeCashflow"))
        total_assets = f(balp.get("totalAssets"))
        total_liab = f(balp.get("totalLiabilities"))
        curr_assets = f(balp.get("totalCurrentAssets"))
        curr_liab = f(balp.get("totalCurrentLiabilities"))
        inventory = f(balp.get("inventory"))
        r_and_d = f(incp.get("researchAndDevelopment"))
        eps = f(incp.get("reportedEPS") or (ov.get("EPS") if isinstance(ov, dict) else np.nan))
        market_cap = f(ov.get("MarketCapitalization")) if isinstance(ov, dict) else np.nan
        pe_ratio = f(ov.get("PERatio")) if isinstance(ov, dict) else np.nan
        pb_ratio = f(ov.get("PriceToBookRatio")) if isinstance(ov, dict) else np.nan
        beta = f(ov.get("Beta")) if isinstance(ov, dict) else np.nan
        dividend_yield = f(ov.get("DividendYield")) if isinstance(ov, dict) else np.nan

        equity = total_assets - total_liab if total_assets and total_liab else np.nan
        debt_to_equity = total_liab / equity if equity else np.nan
        current_ratio = curr_assets / curr_liab if curr_liab else np.nan
        quick_ratio = (curr_assets - inventory) / curr_liab if curr_liab else np.nan
        gross_margin = gross / rev if rev else np.nan
        operating_margin = op_inc / rev if rev else np.nan
        net_margin = net / rev if rev else np.nan
        roe = net / equity if equity else np.nan
        roa = net / total_assets if total_assets else np.nan
        roic = (op_inc * 0.79) / (total_assets - curr_liab) if (total_assets and curr_liab and op_inc) else np.nan
        inventory_turnover = (rev - gross) / inventory if (inventory and rev and gross) else np.nan
        cash_and_equiv = f(balp.get("cashAndCashEquivalentsAtCarryingValue") or cfp.get("cashAndCashEquivalents"))
        enterprise_value = market_cap + total_liab - cash_and_equiv if (market_cap and total_liab) else np.nan
        ev_to_ebitda = enterprise_value / ebitda if (enterprise_value and ebitda) else np.nan
        ps_ratio = market_cap / (rev * 4) if (market_cap and rev) else np.nan
        r_and_d_ratio = (r_and_d / rev) if (r_and_d and rev) else np.nan

        records.append({
            "Symbol": symbol,
            "FiscalDateEnding": date,
            "Revenue": rev,
            "NetIncome": net,
            "GrossProfit": gross,
            "OperatingIncome": op_inc,
            "EBITDA": ebitda,
            "OperatingCashFlow": ocf,
            "CapEX": capex,
            "FreeCashFlow": fcf,
            "TotalAssets": total_assets,
            "TotalLiabilities": total_liab,
            "CurrentAssets": curr_assets,
            "CurrentLiabilities": curr_liab,
            "Inventory": inventory,
            "InventoryTurnover": inventory_turnover,
            "EPS": eps,
            "PERatio": pe_ratio,
            "PBRatio": pb_ratio,
            "MarketCapitalization": market_cap,
            "PS_Ratio": ps_ratio,
            "EV": enterprise_value,
            "EV/EBITDA": ev_to_ebitda,
            "DebtToEquity": debt_to_equity,
            "CurrentRatio": current_ratio,
            "QuickRatio": quick_ratio,
            "ROE": roe,
            "ROA": roa,
            "ROIC": roic,
            "Beta": beta,
            "DividendYield": dividend_yield,
            "GrossMargin": gross_margin,
            "OperatingMargin": operating_margin,
            "NetMargin": net_margin,
            "R&D_to_Revenue": r_and_d_ratio,
        })

    df = pd.DataFrame(records)
    df["FiscalDateEnding"] = pd.to_datetime(df["FiscalDateEnding"])
    df.sort_values("FiscalDateEnding", inplace=True)
    df["RevenueGrowth"] = df["Revenue"].pct_change()
    df["NetIncomeGrowth"] = df["NetIncome"].pct_change()
    return df

if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)
    for s in SYMBOLS:
        try:
            df = fetch_full_history(s)
            path = os.path.join(OUT_DIR, f"{s}_fundamentals_full.csv")
            df.to_csv(path, index=False)
            print(f"💾 {s} kaydedildi → {path}")
            print(f"    Kapsam: {df['FiscalDateEnding'].min().date()} → {df['FiscalDateEnding'].max().date()} ({len(df)} çeyrek)")
        except Exception as e:
            print(f"⚠️ {s} hata: {e}")
        time.sleep(RATE_LIMIT_SLEEP)
    print("\n✅ Tüm şirketler tamamlandı.")


📥 AMD tüm çeyrekler alınıyor...
💾 AMD kaydedildi -> data/AMD_fundamentals_full.csv
    Kapsam: 2005-06-30 → 2025-06-30  (81 çeyrek)
📥 ARM tüm çeyrekler alınıyor...
💾 ARM kaydedildi -> data/ARM_fundamentals_full.csv
    Kapsam: 1998-03-31 → 2025-06-30  (61 çeyrek)
📥 NVDA tüm çeyrekler alınıyor...
💾 NVDA kaydedildi -> data/NVDA_fundamentals_full.csv
    Kapsam: 2005-07-31 → 2025-07-31  (81 çeyrek)
📥 INTC tüm çeyrekler alınıyor...
💾 INTC kaydedildi -> data/INTC_fundamentals_full.csv
    Kapsam: 2005-09-30 → 2025-09-30  (81 çeyrek)
📥 MU tüm çeyrekler alınıyor...
💾 MU kaydedildi -> data/MU_fundamentals_full.csv
    Kapsam: 2005-08-31 → 2025-08-31  (81 çeyrek)

✅ Tüm şirketler tamamlandı.


In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np

load_dotenv()
API_KEY = os.getenv("ALPHA_API_KEY")

BASE = "https://www.alphavantage.co/query"
SYMBOLS = ["AMD", "ARM", "NVDA", "INTC", "MU"]
OUT_DIR = "data/technical"
RATE_LIMIT_SLEEP = 1

def call_av(params, key_name):
    """Alpha Vantage API çağrısı (retry + JSON key kontrolü)"""
    params["apikey"] = API_KEY
    for _ in range(3):
        r = requests.get(BASE, params=params, timeout=40)
        if r.status_code == 200:
            data = r.json()
            if key_name in data:
                return data[key_name]
            if "Note" in data or "Error Message" in data:
                time.sleep(2)
                continue
        time.sleep(2)
    return {}

def get_series(symbol, func, key_name, **extra):
    """Teknik gösterge çekici"""
    params = {"function": func, "symbol": symbol, "datatype": "json"}
    params.update(extra)
    return call_av(params, key_name)

def fetch_technical(symbol):
    print(f"📈 {symbol} teknik göstergeler alınıyor...")

    ts = call_av(
        {"function": "TIME_SERIES_DAILY_ADJUSTED", "symbol": symbol, "outputsize": "full"},
        "Time Series (Daily)"
    )
    if not ts:
        raise ValueError("Zaman serisi alınamadı")

    df = pd.DataFrame(ts).T
    df.index = pd.to_datetime(df.index)
    df.columns = [
        "Open", "High", "Low", "Close", "Adj Close", "Volume",
        "Dividend Amount", "Split Coefficient"
    ][:len(df.columns)]
    df = df[["Open", "High", "Low", "Adj Close", "Volume"]].astype(float)
    df.sort_index(inplace=True)
    df["Symbol"] = symbol

    ema = get_series(symbol, "EMA", "Technical Analysis: EMA", interval="daily", time_period=50, series_type="close")
    if ema:
        df["EMA"] = pd.Series({pd.to_datetime(k): float(v["EMA"]) for k, v in ema.items()})

    rsi = get_series(symbol, "RSI", "Technical Analysis: RSI", interval="daily", time_period=14, series_type="close")
    if rsi:
        df["RSI"] = pd.Series({pd.to_datetime(k): float(v["RSI"]) for k, v in rsi.items()})

    macd = get_series(symbol, "MACD", "Technical Analysis: MACD", interval="daily", series_type="close")
    if macd:
        df["MACD"] = pd.Series({pd.to_datetime(k): float(v["MACD"]) for k, v in macd.items()})
        df["MACD_Signal"] = pd.Series({pd.to_datetime(k): float(v["MACD_Signal"]) for k, v in macd.items()})

    obv = get_series(symbol, "OBV", "Technical Analysis: OBV", interval="daily")
    if obv:
        df["OBV"] = pd.Series({pd.to_datetime(k): float(v["OBV"]) for k, v in obv.items()})

    atr = get_series(symbol, "ATR", "Technical Analysis: ATR", interval="daily", time_period=14)
    if atr:
        df["ATR"] = pd.Series({pd.to_datetime(k): float(v["ATR"]) for k, v in atr.items()})

    bb = get_series(symbol, "BBANDS", "Technical Analysis: BBANDS", interval="daily", time_period=20, series_type="close")
    if bb:
        df["Real Upper Band"] = pd.Series({pd.to_datetime(k): float(v["Real Upper Band"]) for k, v in bb.items()})
        df["Real Lower Band"] = pd.Series({pd.to_datetime(k): float(v["Real Lower Band"]) for k, v in bb.items()})

    stoch = get_series(symbol, "STOCH", "Technical Analysis: STOCH", interval="daily")
    if stoch:
        df["SlowK"] = pd.Series({pd.to_datetime(k): float(v["SlowK"]) for k, v in stoch.items()})
        df["SlowD"] = pd.Series({pd.to_datetime(k): float(v["SlowD"]) for k, v in stoch.items()})

    vwap = get_series(symbol, "VWAP", "Technical Analysis: VWAP", interval="daily")
    if vwap:
        df["VWAP"] = pd.Series({pd.to_datetime(k): float(v["VWAP"]) for k, v in vwap.items()})

    df["Log_Returns"] = np.log(df["Adj Close"] / df["Adj Close"].shift(1))
    df["Volatility"] = df["Log_Returns"].rolling(window=30).std() * np.sqrt(252)

    df.reset_index(inplace=True)
    df.rename(columns={"index": "Date"}, inplace=True)
    df.sort_values("Date", inplace=True)

    return df

if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)
    for s in SYMBOLS:
        try:
            df = fetch_technical(s)
            path = os.path.join(OUT_DIR, f"{s}_technical_full.csv")
            df.to_csv(path, index=False)
            print(f"💾 {s} kaydedildi → {path}")
            print(f"    Kapsam: {df['Date'].min().date()} → {df['Date'].max().date()} ({len(df)} gün)")
        except Exception as e:
            print(f"⚠️ {s} hata: {e}")
        time.sleep(RATE_LIMIT_SLEEP)
    print("\n✅ Tüm teknik veriler tamamlandı.")


📈 AMD teknik göstergeler alınıyor...
💾 AMD kaydedildi → data/technical/AMD_technical_full.csv
    Kapsam: 1999-11-01 → 2025-10-31 (6541 gün)
📈 ARM teknik göstergeler alınıyor...
💾 ARM kaydedildi → data/technical/ARM_technical_full.csv
    Kapsam: 2023-09-14 → 2025-10-31 (536 gün)
📈 NVDA teknik göstergeler alınıyor...
💾 NVDA kaydedildi → data/technical/NVDA_technical_full.csv
    Kapsam: 1999-11-01 → 2025-10-31 (6541 gün)
📈 INTC teknik göstergeler alınıyor...
💾 INTC kaydedildi → data/technical/INTC_technical_full.csv
    Kapsam: 1999-11-01 → 2025-10-31 (6541 gün)
📈 MU teknik göstergeler alınıyor...
💾 MU kaydedildi → data/technical/MU_technical_full.csv
    Kapsam: 1999-11-01 → 2025-10-31 (6541 gün)

✅ Tüm teknik veriler tamamlandı.


In [ ]:
import os

# Colab runtime'a key ekle
os.environ['ALPHA_API_KEY'] = "YOUR_ALPHA_KEY_HERE"

# Mevcut kodun bunu kullanacak şekilde
api_key = os.getenv("ALPHA_API_KEY")
